Student Performance: Three Regression Models

This notebook uses the provided synthetic dataset of 5,000 observations.

Models
1. Simple Linear Regression: predict exam_score from hours_studied.
2. Multiple Linear Regression: predict exam_score from several student variables.
3. Logistic Regression: predict passed (1 = pass, 0 = fail).
The data is synthetic and should only be used for practice.

1. Load and inspect the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("synthetic_student_performance_5000.csv")
print("Dataset shape:", df.shape)
display(df.head())
display(df.info())
print(df.isnull().sum())

2. Simple Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[["hours_studied"]]
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

simple_model = LinearRegression()
simple_model.fit(X_train, y_train)
y_pred = simple_model.predict(X_test)

print("Intercept:", round(simple_model.intercept_, 3))
print("Coefficient for hours_studied:", round(simple_model.coef_[0], 3))
print("R-squared:", round(r2_score(y_test, y_pred), 3))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("RMSE:", round(mean_squared_error(y_test, y_pred) ** 0.5, 3))

plt.scatter(X_test["hours_studied"], y_test, alpha=0.35, label="Actual scores")
order = np.argsort(X_test["hours_studied"].to_numpy())
plt.plot(X_test["hours_studied"].to_numpy()[order], y_pred[order], label="Regression line")
plt.xlabel("Hours studied")
plt.ylabel("Exam score")
plt.title("Simple Linear Regression")
plt.legend()
plt.show()

How to interpret it: The coefficient tells you how much the predicted exam score changes, on average, for one extra hour studied. R-squared measures how much variation in exam scores the model explains. MAE and RMSE measure prediction error; lower values are generally better.

3. Multiple Linear Regression

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

features = [
    "hours_studied", "attendance_percent", "previous_score",
    "assignments_completed", "sleep_hours", "internet_access"
]
X = df[features]
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    [("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), ["internet_access"])],
    remainder="passthrough"
)

multiple_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])
multiple_model.fit(X_train, y_train)
y_pred = multiple_model.predict(X_test)

print("R-squared:", round(r2_score(y_test, y_pred), 3))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 3))
print("RMSE:", round(mean_squared_error(y_test, y_pred) ** 0.5, 3))

How to interpret it: This model uses several predictors at the same time. R-squared, MAE, and RMSE can be compared with the simple model to see whether the extra variables improve prediction on the test data. internet_access is categorical, so the pipeline converts it to a numerical representation automaticall

4. Logistic Regression

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

features = [
    "hours_studied", "attendance_percent", "previous_score",
    "assignments_completed", "sleep_hours", "internet_access"
]
X = df[features]
y = df["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), ["internet_access"]),
    ("numeric", StandardScaler(), [
        "hours_studied", "attendance_percent", "previous_score",
        "assignments_completed", "sleep_hours"
    ])
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
y_probability = logistic_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_probability), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

ModuleNotFoundError: No module named 'sklearn'

How to interpret it: Logistic regression predicts the probability of a student passing. The default decision threshold is 0.5: probabilities at least 0.5 are classified as pass (1), and lower probabilities as fail (0). Accuracy is the proportion of correct predictions. The confusion matrix shows correct and incorrect predictions by class. ROC-AUC measures how well the model separates the two classes.

5. Try a prediction with new values

In [2]:
# Example: predict the exam score for a student who studied for 6 hours
new_student_simple = pd.DataFrame({"hours_studied": [6]})
print("Simple regression predicted exam score:",
      round(simple_model.predict(new_student_simple)[0], 2))

# Example for multiple linear regression and logistic regression
new_student = pd.DataFrame({
    "hours_studied": [6],
    "attendance_percent": [85],
    "previous_score": [65],
    "assignments_completed": [8],
    "sleep_hours": [7],
    "internet_access": ["Yes"]
})

print("Multiple regression predicted exam score:",
      round(multiple_model.predict(new_student)[0], 2))
print("Probability of passing:",
      round(logistic_model.predict_proba(new_student)[0, 1], 3))
print("Predicted pass/fail (1 = pass, 0 = fail):",
      logistic_model.predict(new_student)[0])

NameError: name 'pd' is not defined

Important note
The passed column is a separately generated synthetic outcome. It does not necessarily match a rule such as exam_score >= 50. If your lecturer expects pass/fail to be defined by a pass mark, create the target from exam_score using the pass mark specified in your assignment, then rerun logistic regression.

5. Try a prediction with new values

In [ ]:
# Example: predict the exam score for a student who studied for 6 hours
new_student_simple = pd.DataFrame({"hours_studied": [6]})
print("Simple regression predicted exam score:",
      round(simple_model.predict(new_student_simple)[0], 2))

# Example for multiple linear regression and logistic regression
new_student = pd.DataFrame({
    "hours_studied": [6],
    "attendance_percent": [85],
    "previous_score": [65],
    "assignments_completed": [8],
    "sleep_hours": [7],
    "internet_access": ["Yes"]
})

print("Multiple regression predicted exam score:",
      round(multiple_model.predict(new_student)[0], 2))
print("Probability of passing:",
      round(logistic_model.predict_proba(new_student)[0, 1], 3))
print("Predicted pass/fail (1 = pass, 0 = fail):",
      logistic_model.predict(new_student)[0])

Important note
The passed column is a separately generated synthetic outcome. It does not necessarily match a rule such as exam_score >= 50. If your lecturer expects pass/fail to be defined by a pass mark, create the target from exam_score using the pass mark specified in your assignment, then rerun logistic regression.